In [ ]:
import pandas as pd
import numpy as np
import os
print(os.getcwd())

In [2]:
income_df = pd.read_csv('income_by_nta.csv')
bus_subway_df = pd.read_csv('bus_vs_subway_by_nta.csv')
vehicles_df = pd.read_csv('vehicles_stored_by_NTA.csv')

In [3]:
# car score
vehicles_df = vehicles_df.rename(columns={'NTA': 'NTACode'})
vehicles_df = vehicles_df[['NTACode', '% Use Car']].rename(columns={'% Use Car': 'pct_use_car'})
vehicles_df['pct_use_car'] = pd.to_numeric(vehicles_df['pct_use_car'], errors='coerce')
 
low  = vehicles_df['pct_use_car'].min()
high = vehicles_df['pct_use_car'].max()
vehicles_df['car_score'] = 1 - ((vehicles_df['pct_use_car'] - low) / (high - low))

In [4]:
route_nta = pd.read_csv('route_nta_mapping.csv')
route_nta = route_nta.dropna(subset=['NTACode'])
route_nta['route_id'] = route_nta['route_id'].astype(str)

In [5]:
# reliability score
lateness_files = [
    'weighted_lateness_bronx.csv',
    'weighted_lateness_brooklyn.csv',
    'weighted_lateness_manhattan.csv',
    'weighted_lateness_queens.csv',
    'weighted_lateness_si.csv'
]
 
lateness_parts = []
for file in lateness_files:
    if os.path.exists(file):
        df = pd.read_csv(file)
        df.columns = ['route_id', 'weighted_avg_lateness']
        lateness_parts.append(df)
    else:
        print(f"WARNING: {file} not found, skipping")
 
lateness_df = pd.concat(lateness_parts, ignore_index=True)
 
lateness_df['weighted_avg_lateness'] = lateness_df['weighted_avg_lateness'].abs()
lateness_df['route_id'] = lateness_df['route_id'].astype(str)
 
#attach lateness to each route-NTA pair
lateness_nta = route_nta.merge(lateness_df, on='route_id', how='inner')
 
#average lateness across all routes serving each NTA
nta_lateness = (
    lateness_nta
    .groupby('NTACode')['weighted_avg_lateness']
    .mean()
    .reset_index()
    .rename(columns={'weighted_avg_lateness': 'avg_lateness'})
)
 
low  = nta_lateness['avg_lateness'].min()
high = nta_lateness['avg_lateness'].max()
nta_lateness['reliability_score'] = (1 - (nta_lateness['avg_lateness'] - low) / (high - low))

print(f"NTAs with reliability scores: {len(nta_lateness)}")

NTAs with reliability scores: 211


In [6]:
#ridership
ridership_df = pd.read_csv('MTA_Bus_Route_Averages_Clean.csv')
ridership_df = ridership_df[['bus_route', 'avg_ridership']].rename(columns={'bus_route': 'route_id'})
ridership_df['route_id'] = ridership_df['route_id'].astype(str)
 
ridership_df = ridership_df[ridership_df['avg_ridership'] > 0]
 
# Sanity check route ID format matches
print("Sample ridership route IDs:", ridership_df['route_id'].head(5).tolist())
print("Sample route_nta route IDs:", route_nta['route_id'].head(5).tolist())
 
# Attach ridership to each route-NTA pair
ridership_nta = route_nta.merge(ridership_df, on='route_id', how='inner')
 
# Average ridership across all routes serving each NTA
nta_ridership = (
    ridership_nta
    .groupby('NTACode')['avg_ridership']
    .mean()
    .reset_index()
    .rename(columns={'avg_ridership': 'avg_route_ridership'})
)
 
low  = nta_ridership['avg_route_ridership'].min()
high = nta_ridership['avg_route_ridership'].max()

nta_ridership['ridership_score'] = (
    (nta_ridership['avg_route_ridership'] - low) / (high - low)
)
 
print(f"NTAs with ridership scores: {len(nta_ridership)}")

Sample ridership route IDs: ['B1', 'B100', 'B103', 'B11', 'B12']
Sample route_nta route IDs: ['B82+', 'B82+', 'B82+', 'B82+', 'B82+']
NTAs with ridership scores: 215


In [7]:
merged = income_df[['NTACode', 'income_score']]
 
merged = merged.merge(
    vehicles_df[['NTACode', 'car_score']],
    on='NTACode', how='outer'
)
merged = merged.merge(
    bus_subway_df[['NTACode', 'bus_dependency_ratio']].rename(
        columns={'bus_dependency_ratio': 'bus_subway_score'}
    ),
    on='NTACode', how='outer'
)
merged = merged.merge(
    nta_lateness[['NTACode', 'reliability_score']],
    on='NTACode', how='outer'
)
merged = merged.merge(
    nta_ridership[['NTACode', 'ridership_score']],
    on='NTACode', how='outer'
)

In [8]:
# bus need index
# weights are arbitrary
W_INCOME      = 0.25
W_CAR         = 0.20
W_BUS_SUBWAY  = 0.20
W_RELIABILITY = 0.15
W_RIDERSHIP   = 0.20
 
available_weight = W_INCOME + W_CAR + W_BUS_SUBWAY + W_RELIABILITY + W_RIDERSHIP  # 1.0
 
merged['index_score'] = (
    W_INCOME      * merged['income_score'].fillna(0)      +
    W_CAR         * merged['car_score'].fillna(0)         +
    W_BUS_SUBWAY  * merged['bus_subway_score'].fillna(0)  +
    W_RELIABILITY * merged['reliability_score'].fillna(0) +
    W_RIDERSHIP   * merged['ridership_score'].fillna(0)
).round(6)

In [9]:
# sanity check
cols = ['income_score', 'car_score', 'bus_subway_score', 'reliability_score', 'ridership_score']
complete = merged[cols].notna().all(axis=1).sum()
 
print(f"\nTotal NTAs: {len(merged)}")
print(f"NTAs with all 5 scores present: {complete}")
print(f"NTAs missing at least one score: {len(merged) - complete}")
 
print("\nTop 10 highest need NTAs:")
print(
    merged.sort_values('index_score', ascending=False)
    .head(10)[['NTACode'] + cols + ['index_score']]
    .to_string(index=False)
)
 
print("\nBottom 10 lowest need NTAs:")
print(
    merged.sort_values('index_score')
    .head(10)[['NTACode'] + cols + ['index_score']]
    .to_string(index=False)
)



Total NTAs: 262
NTAs with all 5 scores present: 175
NTAs missing at least one score: 87

Top 10 highest need NTAs:
NTACode  income_score  car_score  bus_subway_score  reliability_score  ridership_score  index_score
 MN1101      0.938590   0.944516            0.9423           0.734103         0.937810     0.909688
 MN0902      0.879181   0.948387            0.9744           0.811577         0.789928     0.884075
 BX0601      1.000000   0.752258            0.9394           0.744520         0.784556     0.856921
 MN0903      0.804415   0.948387            0.9538           0.771137         0.778937     0.852999
 BK1101      0.776816   0.748387            0.9306           0.756043         1.000000     0.843408
 BK1303      0.872996   0.763871            0.9744           0.850785         0.748409     0.843203
 BK1401      0.727082   0.827097            0.9718           0.807594         0.888658     0.840421
 BK1701      0.730150   0.834839            0.9444           0.836595         0.8818

In [10]:
# export final dataset
#only export NTAs with at least some real data
merged_clean = merged[merged[cols].notna().any(axis=1)]
 
merged_clean[['NTACode', 'index_score']].rename(
    columns={'index_score': 'Score'}
).to_csv('bus_need_index_final.csv', index=False)
 
merged.to_csv('bus_need_index_final_full.csv', index=False)
 
print("\nExported:")
print("  bus_need_index_final.csv      → upload to map")
print("  bus_need_index_final_full.csv → full dataset for group")


Exported:
  bus_need_index_final.csv      → upload to map
  bus_need_index_final_full.csv → full dataset for group


In [ ]:
# route to NTA mapping extraction from GTFS data to connect lateness data to NTAs
gtfs_folders = [
    '../nta_bus_mapping/data/gtfs_b',
    '../nta_bus_mapping/data/gtfs_bx',
    '../nta_bus_mapping/data/gtfs_m',
    '../nta_bus_mapping/data/gtfs_q',
    '../nta_bus_mapping/data/gtfs_si'
]

trips_list = []
stop_times_list = []

for folder in gtfs_folders:
    trips_path = os.path.join(folder, 'trips.txt')
    stop_times_path = os.path.join(folder, 'stop_times.txt')
    if os.path.exists(trips_path):
        trips_list.append(pd.read_csv(trips_path, usecols=['route_id', 'trip_id']))
    if os.path.exists(stop_times_path):
        stop_times_list.append(pd.read_csv(stop_times_path, usecols=['trip_id', 'stop_id']))

all_trips = pd.concat(trips_list, ignore_index=True).drop_duplicates()
all_stop_times = pd.concat(stop_times_list, ignore_index=True).drop_duplicates()

stops_df = pd.read_csv('../nta_bus_mapping/stops_with_nta.csv')
stops_df['stop_id'] = stops_df['stop_id'].astype(str)

route_stops = all_trips.merge(all_stop_times, on='trip_id')
route_stops = route_stops[['route_id', 'stop_id']].drop_duplicates()
route_stops['stop_id'] = route_stops['stop_id'].astype(str)

route_nta = route_stops.merge(
    stops_df[['stop_id', 'NTACode']], on='stop_id', how='inner'
)[['route_id', 'NTACode']].drop_duplicates()

route_nta.to_csv('route_nta_mapping.csv', index=False)
print(f"Done. Exported {len(route_nta)} route-NTA pairs.")

In [12]:
import json

component_cols = ['income_score', 'car_score', 'bus_subway_score', 'reliability_score', 'ridership_score']

component_dict = {}
for _, row in merged_clean.iterrows():
    code = row['NTACode']
    component_dict[code] = {
        'income':      round(row['income_score'],    6) if pd.notna(row['income_score'])      else 0,
        'car':         round(row['car_score'],       6) if pd.notna(row['car_score'])         else 0,
        'bus_subway':  round(row['bus_subway_score'],6) if pd.notna(row['bus_subway_score'])  else 0,
        'reliability': round(row['reliability_score'],6) if pd.notna(row['reliability_score']) else 0,
        'ridership':   round(row['ridership_score'], 6) if pd.notna(row['ridership_score'])   else 0,
    }

with open('../nta_map/component_scores.json', 'w') as f:
    json.dump(component_dict, f)

print(f"Exported component scores for {len(component_dict)} NTAs")

Exported component scores for 237 NTAs
